In [0]:
# ==========================================================
# GOLD LAYER: Final analytics tables
#
# What happens here:
# - Read cleaned data from Silver
# - Create one main fact table
# - Build business-level aggregations (KPIs)
#
# These tables are directly used for dashboards / BI
# ==========================================================

from pyspark.sql.functions import (
    col, count, countDistinct, sum as _sum, avg as _avg,
    max as _max, min as _min, round as _round, when,
    coalesce, lit, datediff
)

print("Building Gold layer tables...")


# --- Load Silver tables ---
# These are already cleaned and ready to use
orders = spark.table("silver.orders")
items = spark.table("silver.order_items")
payments = spark.table("silver.order_payments")
reviews = spark.table("silver.order_reviews")


# ==========================================================
# GOLD TABLE 1: fact_order_items (main table)
# ==========================================================
print("\n--- gold.fact_order_items ---")

# Aggregate payments at order level
# (one order can have multiple payment rows)
payment_agg = payments.groupBy("order_id").agg(
    _sum("payment_value").alias("payment_value"),
    _max("payment_type").alias("payment_type"),
    _avg("payment_installments").alias("avg_installments")
)

# Aggregate reviews at order level
review_agg = reviews.groupBy("order_id").agg(
    _avg("review_score").alias("review_score")
)

# Build main fact table by joining everything
fact = (
    items.alias("oi")
    .join(orders.alias("o"), "order_id")
    .join(payment_agg.alias("pa"), "order_id", "left")
    .join(review_agg.alias("r"), "order_id", "left")
    .select(
        col("oi.order_id"),
        col("oi.order_item_id"),
        col("o.customer_id"),
        col("oi.product_id"),
        col("oi.seller_id"),

        # Pricing details
        col("oi.price"),
        col("oi.freight_value"),
        (col("oi.price") + col("oi.freight_value")).alias("total_amount"),

        # Payment info
        col("pa.payment_value"),
        col("pa.payment_type"),
        col("pa.avg_installments"),

        # Order details
        col("o.order_status"),
        col("o.order_purchase_timestamp").alias("purchase_timestamp"),
        col("o.order_delivered_customer_date").alias("delivered_date"),
        col("o.order_estimated_delivery_date"),

        # Review (default 0 if missing)
        coalesce(col("r.review_score"), lit(0)).alias("review_score"),

        # Delivery metrics
        datediff(
            col("o.order_delivered_customer_date"),
            col("o.order_purchase_timestamp")
        ).alias("delivery_days"),

        # Flags for KPI calculation
        when(
            col("o.order_delivered_customer_date") >
            col("o.order_estimated_delivery_date"), 1
        ).otherwise(0).alias("delay_flag"),

        when(
            col("o.order_delivered_customer_date") <=
            col("o.order_estimated_delivery_date"), 1
        ).otherwise(0).alias("on_time_flag")
    )
)

# Save fact table
fact.write.format("delta").mode("overwrite").saveAsTable("gold.fact_order_items")

print(f"Rows: {fact.count():,}")


# ==========================================================
# GOLD TABLE 2: order_summary (per order view)
# ==========================================================
print("\n--- gold.order_summary ---")

fact = spark.table("gold.fact_order_items")

order_summary = fact.groupBy("order_id").agg(
    count("order_item_id").alias("total_items"),
    _round(_sum("price"), 2).alias("total_price"),
    _round(_sum("freight_value"), 2).alias("total_freight"),
    _round(_sum("total_amount"), 2).alias("total_amount"),

    # Order-level info
    _max("order_status").alias("order_status"),
    _min("purchase_timestamp").alias("purchase_date"),
    _max("delivered_date").alias("delivered_date"),

    # Average delivery time
    _round(_avg("delivery_days"), 0).cast("int").alias("delivery_days")
)

order_summary.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.order_summary")

print(f"Rows: {order_summary.count():,}")


# ==========================================================
# GOLD TABLE 3: customer_metrics (CLV + behavior)
# ==========================================================
print("\n--- gold.customer_metrics ---")

customer_metrics = fact.groupBy("customer_id").agg(
    countDistinct("order_id").alias("total_orders"),
    _round(_sum("total_amount"), 2).alias("total_spent"),
    _round(_avg("total_amount"), 2).alias("avg_order_value"),
    _max("purchase_timestamp").alias("last_order_date"),
    _round(_avg("review_score"), 1).alias("avg_review_score")
)

customer_metrics.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.customer_metrics")

print(f"Rows: {customer_metrics.count():,}")


# ==========================================================
# GOLD TABLE 4: seller_performance (business KPIs)
# ==========================================================
print("\n--- gold.seller_performance ---")

seller_perf = fact.groupBy("seller_id").agg(
    countDistinct("order_id").alias("total_orders"),
    _round(_sum("price"), 2).alias("total_revenue"),
    _round(_avg("review_score"), 2).alias("avg_review_score"),

    # % of on-time deliveries
    _round(
        (_sum("on_time_flag") * 100.0 / count("*")), 1
    ).alias("on_time_delivery_rate"),

    _round(_avg("delivery_days"), 1).alias("avg_delivery_days")
)

seller_perf.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.seller_performance")

print(f"Rows: {seller_perf.count():,}")


# ==========================================================
# GOLD TABLE 5: product_performance
# ==========================================================
print("\n--- gold.product_performance ---")

product_perf = fact.groupBy("product_id").agg(
    count("*").alias("total_quantity"),
    _round(_sum("price"), 2).alias("total_sales"),
    _round(_avg("price"), 2).alias("avg_price"),
    _round(_avg("review_score"), 2).alias("avg_review_score")
)

product_perf.write.format("delta").mode("overwrite") \
    .saveAsTable("gold.product_performance")

print(f"Rows: {product_perf.count():,}")


# ==========================================================
# Done
# ==========================================================
print("\n" + "="*60)
print("Gold layer build COMPLETE!")
print("1. fact_order_items  → main table")
print("2. order_summary     → order-level view")
print("3. customer_metrics  → CLV + behavior")
print("4. seller_performance → seller KPIs")
print("5. product_performance → product insights")
print("="*60)